# Toy Binary Classifier with a Custom Autograd Engine

This notebook trains a tiny MLP on a simple binary classification dataset using the gradients implemented in `src/engine.py` and `src/nn.py`. The goal is to show the same reverse-mode training loop that frameworks like PyTorch use, but in a transparent, educational form.

In [ ]:
import random

from src.engine import Value
from src.nn import Linear, ReLU, MSELoss

random.seed(0)

X = [
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0],
]
y = [0.0, 1.0, 1.0, 0.0]

class MLP:
    def __init__(self):
        self.hidden = Linear(2, 4)
        self.output = Linear(4, 1)

    def forward(self, x):
        hidden_values = self.hidden.forward([Value(v, label=f"x{idx}") for idx, v in enumerate(x)])
        activated = ReLU().forward(hidden_values)
        prediction = self.output.forward(activated)
        return prediction[0]

model = MLP()
criterion = MSELoss()

for epoch in range(300):
    total_loss = Value(0.0)
    for sample, target in zip(X, y):
        pred = model.forward(sample)
        total_loss = total_loss + (pred - Value(target)) * (pred - Value(target))
    total_loss = total_loss / len(X)
    total_loss.backward()

    for layer in [model.hidden, model.output]:
        for parameter in layer.weight:
            for value in parameter:
                value.data -= 0.05 * value.grad
                value.grad = 0.0
        if layer.bias is not None:
            for value in layer.bias:
                value.data -= 0.05 * value.grad
                value.grad = 0.0

print('final training loss:', total_loss.data)
for sample, target in zip(X, y):
    prediction = model.forward(sample)
    print(sample, '->', prediction.data, 'target', target)